In [2]:
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import torch
import torch.nn as nn
import timm
import torch.optim as optim

# ==========================================
# 1. MODUL ATTENTION (TETAP SAMA)
# ==========================================
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x))

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.ca(x)
        return out * self.sa(out)

# ==========================================
# 2. ARSITEKTUR UTAMA (YANG DIPERBAIKI)
# ==========================================
class MobileNetV4_CBAM(nn.Module):
    def __init__(self, num_classes=2):
        super(MobileNetV4_CBAM, self).__init__()
        
        # A. Panggil Backbone
        self.backbone = timm.create_model('mobilenetv4_conv_medium', pretrained=True, num_classes=0, global_pool='')
        
        # B. [PERBAIKAN] Deteksi Channel Otomatis!
        # Kita masukkan 1 gambar acak (dummy) untuk melihat wujud asli outputnya
        dummy_input = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            dummy_output = self.backbone(dummy_input)
            
        feature_dim = dummy_output.shape[1] # Pasti akan terbaca 1280
        print(f"✨ Deteksi Otomatis: Channel output sebenarnya adalah {feature_dim}")
        
        # C. Pasang CBAM dan Classifier dengan ukuran yang benar
        self.cbam = CBAM(feature_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ==========================================
# 3. INISIALISASI ULANG MODEL & OPTIMIZER
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_proposed = MobileNetV4_CBAM(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
# Harus didefinisikan ulang di sini karena 'model_proposed' baru saja diganti "jeroannya"
optimizer_proposed = optim.AdamW(model_proposed.parameters(), lr=0.0001)

print("✅ Model Anti-Error Siap Dijalankan!")

✨ Deteksi Otomatis: Channel output sebenarnya adalah 1280
✅ Model Anti-Error Siap Dijalankan!


In [9]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Tentukan Path Dataset (Sesuaikan dengan lokasi folder di laptopmu)
base_dir = "./DATASET/real_vs_fake/real-vs-fake"

# 2. Pengaturan Transformasi Gambar
# Untuk Latih (Train) kita tambahkan sedikit variasi agar AI lebih pintar
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Menambah variasi
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Untuk Validasi & Test (Cukup resize dan normalisasi saja)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Menghubungkan Kodingan ke Folder
train_dataset = datasets.ImageFolder(os.path.join(base_dir, 'train'), train_transforms)
valid_dataset = datasets.ImageFolder(os.path.join(base_dir, 'valid'), val_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(base_dir, 'test'), val_transforms)

# 4. Membuat DataLoader (Jembatan Data)
# Mengingat VRAM 1660 Super adalah 6GB, Batch Size 32 adalah angka yang aman
batch_size = 32

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2),
    'valid': DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2),
    'test':  DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
}

print(f"✅ Dataloader Berhasil Dibuat!")
print(f"Total Gambar Latih: {len(train_dataset):,}")
print(f"Total Gambar Validasi: {len(valid_dataset):,}")
print(f"Total Gambar Uji: {len(test_dataset):,}")

✅ Dataloader Berhasil Dibuat!
Total Gambar Latih: 100,000
Total Gambar Validasi: 20,000
Total Gambar Uji: 20,000


In [12]:
import time
import copy
import os
import sys
from tqdm.notebook import tqdm # Kita pakai tqdm.notebook biar rapi
import torch.optim as optim

# 1. Tentukan Loss dan Optimizer BARU untuk model CBAM
criterion = torch.nn.CrossEntropyLoss()
optimizer_proposed = optim.AdamW(model_proposed.parameters(), lr=0.0001)

# 2. Fungsi Training Khusus CBAM
def train_model_cbam(model, criterion, optimizer, num_epochs=5):
    since = time.time()
    best_acc = 0.0
    
    if not os.path.exists('checkpoints'):
        os.makedirs('checkpoints')

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'valid']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Bar Progres
            pbar = tqdm(dataloaders[phase], desc=f"{phase.upper()}", dynamic_ncols=True)
            
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f'{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Simpan CHECKPOINT KHUSUS CBAM
            if phase == 'valid':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), 'checkpoints/best_mobilenetv4_cbam_model.pth')
                    print("🌟 Model CBAM Terbaik Disimpan!")

    time_elapsed = time.time() - since
    print(f'\nTraining CBAM selesai dalam {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Akurasi Terbaik (Val): {best_acc:.4f}')
    
    return model

# 3. MULAI TRAINING (Tarik napas, siapkan GPU-mu!)
model_proposed = train_model_cbam(model_proposed, criterion, optimizer_proposed, num_epochs=5)


Epoch 1/5
----------


TRAIN:   0%|          | 0/3125 [00:00<?, ?it/s]

TRAIN Loss: 0.0992 Acc: 0.9594


VALID:   0%|          | 0/625 [00:00<?, ?it/s]

VALID Loss: 0.0341 Acc: 0.9866
🌟 Model CBAM Terbaik Disimpan!

Epoch 2/5
----------


...

VALID Loss: 0.0170 Acc: 0.9939
🌟 Model CBAM Terbaik Disimpan!

Epoch 5/5
----------


TRAIN:   0%|          | 0/3125 [00:00<?, ?it/s]

TRAIN Loss: 0.0145 Acc: 0.9947


VALID:   0%|          | 0/625 [00:00<?, ?it/s]

VALID Loss: 0.0103 Acc: 0.9965
🌟 Model CBAM Terbaik Disimpan!

Training CBAM selesai dalam 46m 59s
Akurasi Terbaik (Val): 0.9965
